# notebook_dna_load

Loads Ancestry DNA match data from Fivetran-managed staging tables into `silver_dna_match`.

## Workflow
1. Export Ancestry match pages as HTML (Chrome, Android)
2. Parse HTML → CSV using ChatGPT prompt (see ChatGPT Prompts tab in DNA_Research_Log)
3. Save CSV to Google Drive folder synced by Fivetran
4. Fivetran loads CSV into staging table automatically
5. Run this notebook to upsert from staging → silver_dna_match

## Staging tables (one per kit)
| kit_id | staging table |
|---|---|
| `ed_ancestry` | `staging_google_drive.dna_matches_ancestry_ed` |
| `dad_ancestry` | `staging_google_drive.dna_matches_ancestry_dad` |
| `gillian_ancestry` | `staging_google_drive.dna_matches_ancestry_gillian` |

MyHeritage matches will use separate tables with different column sets — handled in a future notebook.

## Upsert logic
- New matches → INSERT
- Existing matches → UPDATE `shared_cm`, `predicted_relationship`, `side`,
  `has_common_ancestor_hint`, `tree_info`, `tree_people_count`, `managed_by`, `last_seen_date`
- Matches previously seen but absent from staging table → `active = FALSE`

Merge key: `kit_id` + `match_name`

Note: if a match changes their Ancestry display name, the old record is set
`active = FALSE` and a new row is inserted. Acceptable for now — revisit if
name changes cause significant duplicate issues.


In [0]:
# Kit ID → staging table mapping.
# Add new kits here as additional tables are created in Fivetran.
# MyHeritage will be handled in a separate notebook (different column set).
KIT_TABLES = {
    "ed_ancestry":      "staging_google_drive.dna_matches_ancestry_ed",
    "dad_ancestry":     "staging_google_drive.dna_matches_ancestry_dad",
    "gillian_ancestry": "staging_google_drive.dna_matches_ancestry_gillian",
}

# Only process kits whose staging table currently exists.
# This allows the notebook to run safely before Dad/Gillian tables are created.
available_kits = {}
for kit_id, table_name in KIT_TABLES.items():
    try:
        spark.table(table_name).limit(1).count()
        available_kits[kit_id] = table_name
        print(f"  found: {kit_id} → {table_name}")
    except Exception:
        print(f"  skipped (table not found): {table_name}")

if not available_kits:
    raise RuntimeError("No staging tables found — check Fivetran sync has run")


In [0]:
from pyspark.sql import functions as F
from datetime import date

today = date.today().isoformat()

def normalise_ancestry_staging(kit_id, table_name):
    """
    Read one Fivetran-managed Ancestry staging table and normalise to
    the silver_dna_match schema.

    Fivetran column notes:
      - shared_dna_c_m_ : mangled from 'Shared DNA (cM only)', bigint
      - common_ancestor  : 'Yes' / 'No' string
      - _fivetran_synced : metadata timestamp — ignored
      - _file, _line     : Fivetran file metadata — ignored
      - _modified        : file modification timestamp — ignored
    """
    return (
        spark.table(table_name)
        .withColumn("kit_id",    F.lit(kit_id))
        .withColumn("shared_cm", F.col("shared_dna_c_m_").cast("double"))
        .withColumn(
            "has_common_ancestor_hint",
            F.when(F.upper(F.col("common_ancestor")) == "YES", True).otherwise(False)
        )
        .withColumn("last_seen_date", F.lit(today))
        .withColumn("active",         F.lit(True))
        .select(
            "kit_id",
            "match_name",
            "shared_cm",
            "side",
            "predicted_relationship",
            "has_common_ancestor_hint",
            "tree_info",
            F.col("tree_people_count").cast("int").alias("tree_people_count"),
            "avatar_type",
            "managed_by",
            "last_seen_date",
            "active",
        )
        # Drop rows with no match name (blank rows at end of CSV)
        .where(F.col("match_name").isNotNull() & (F.col("match_name") != ""))
    )

# Union all available kit tables into a single incoming DataFrame
kit_dfs = [normalise_ancestry_staging(kit_id, table_name)
           for kit_id, table_name in available_kits.items()]

incoming = kit_dfs[0]
for df in kit_dfs[1:]:
    incoming = incoming.unionByName(df)


counts = incoming.groupBy("kit_id").count().orderBy("kit_id")
print("Incoming rows per kit:")
counts.show()


In [0]:
# Create silver_dna_match if this is the first run.
# On subsequent runs the MERGE below handles everything.
spark.sql("""
    CREATE TABLE IF NOT EXISTS genealogy.silver_dna_match (
        kit_id                   STRING  COMMENT 'e.g. ed_ancestry, dad_ancestry, gillian_ancestry',
        match_name               STRING  COMMENT 'Ancestry username or display name',
        shared_cm                DOUBLE  COMMENT 'Shared centimorgans from platform',
        side                     STRING  COMMENT 'Maternal / Paternal / Unassigned',
        predicted_relationship   STRING  COMMENT 'Ancestry predicted relationship label',
        has_common_ancestor_hint BOOLEAN COMMENT 'True if Ancestry shows a common ancestor hint',
        tree_info                STRING  COMMENT 'No trees / Unlinked tree / Public linked tree etc.',
        tree_people_count        INT     COMMENT 'Number of people in match tree if available',
        avatar_type              STRING  COMMENT 'Male / Female / Photo',
        managed_by               STRING  COMMENT 'Manager name if kit is managed by someone else',
        first_seen_date          STRING  COMMENT 'Date first loaded into this table (ISO 8601)',
        last_seen_date           STRING  COMMENT 'Date last present in a scrape for this kit (ISO 8601)',
        active                   BOOLEAN COMMENT 'False if match no longer appears in scrape exports'
    )
    USING DELTA
    COMMENT 'Raw scraped Ancestry DNA match lists. One row per kit + match_name. Upserted on each load.'
""")
print("Table ready")


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "genealogy.silver_dna_match")

# MERGE on kit_id + match_name.
# Matched rows: update mutable fields + last_seen_date, reactivate if previously
#   deactivated, keep original first_seen_date.
# New rows: insert with first_seen_date = last_seen_date = today.
(
    target.alias("t")
    .merge(
        incoming.alias("s"),
        "t.kit_id = s.kit_id AND t.match_name = s.match_name"
    )
    .whenMatchedUpdate(set={
        "shared_cm":                "s.shared_cm",
        "side":                     "s.side",
        "predicted_relationship":   "s.predicted_relationship",
        "has_common_ancestor_hint": "s.has_common_ancestor_hint",
        "tree_info":                "s.tree_info",
        "tree_people_count":        "s.tree_people_count",
        "avatar_type":              "s.avatar_type",
        "managed_by":               "s.managed_by",
        "last_seen_date":           "s.last_seen_date",
        "active":                   "true",
    })
    .whenNotMatchedInsert(values={
        "kit_id":                   "s.kit_id",
        "match_name":               "s.match_name",
        "shared_cm":                "s.shared_cm",
        "side":                     "s.side",
        "predicted_relationship":   "s.predicted_relationship",
        "has_common_ancestor_hint": "s.has_common_ancestor_hint",
        "tree_info":                "s.tree_info",
        "tree_people_count":        "s.tree_people_count",
        "avatar_type":              "s.avatar_type",
        "managed_by":               "s.managed_by",
        "first_seen_date":          "s.last_seen_date",
        "last_seen_date":           "s.last_seen_date",
        "active":                   "true",
    })
    .execute()
)

print("Merge complete")


In [0]:
# Any active match for a kit that was NOT in the staging table gets marked inactive.
# Scoped to kits present in this run only — other kits are untouched.
# This handles matches that have been removed from Ancestry (deleted account,
# privacy change etc.) — we retain the row but flag it inactive.

kits_in_run = [row["kit_id"] for row in incoming.select("kit_id").distinct().collect()]
kits_sql    = ", ".join(f"'{k}'" for k in kits_in_run)
print(f"Kits in this run: {kits_in_run}")

incoming_keys = incoming.select("kit_id", "match_name")

(
    target.alias("t")
    .merge(
        incoming_keys.alias("s"),
        "t.kit_id = s.kit_id AND t.match_name = s.match_name"
    )
    .whenNotMatchedBySourceAnd(
        f"t.active = true AND t.kit_id IN ({kits_sql})"
    )
    .updateSet({"active": "false"})
    .execute()
)

print("Inactive flag update complete")


In [0]:
print("silver_dna_match row counts:")
(
    spark.table("genealogy.silver_dna_match")
    .groupBy("kit_id", "active")
    .count()
    .orderBy("kit_id", "active")
    .show()
)
